# 01 · Get the data

Downloads every raw series and saves a copy in `data/raw/`.

- 14 currency pairs and the S&P 500 from Yahoo Finance
- 3-month and 10-year interest rates for seven currencies, and the VIX, from FRED

At the end, the date of the download is saved, because Yahoo sometimes revises past data.

In [ ]:
import os
from datetime import date

import pandas as pd
import yfinance as yf

import helpers

os.makedirs(helpers.RAW, exist_ok=True)

## 1. Currency pairs

In [ ]:
for p in helpers.ALL_PAIRS:
    df = yf.download(p + "=X", start=helpers.START, progress=False)
    if len(df) == 0:
        print(p, "NO DATA")
    else:
        df.to_csv(f"{helpers.RAW}/{p}.csv")
        print(p, len(df), "rows,", df.index.min().date(), "to", df.index.max().date())

## 2. S&P 500 (used only to find equity crashes)

In [ ]:
sp = yf.download("^GSPC", start=helpers.START, progress=False)
sp.to_csv(f"{helpers.RAW}/SP500.csv")
print("SP500", len(sp), "rows,", sp.index.min().date(), "to", sp.index.max().date())

## 3. FRED series

In [ ]:
for name, code in helpers.FRED_SERIES.items():
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={code}"
    df = pd.read_csv(url, index_col=0, parse_dates=True)
    df.to_csv(f"{helpers.RAW}/FRED_{name}.csv")
    values = pd.to_numeric(df.iloc[:, 0], errors="coerce").dropna()
    print(name, code, values.index.min().date(), "to", values.index.max().date())

## 4. Record the download date

In [ ]:
with open(f"{helpers.RAW}/download_date.txt", "w") as f:
    f.write(str(date.today()))

print("Downloaded on", date.today())

## 5. Checks

In [ ]:
files = os.listdir(helpers.RAW)
expected = len(helpers.ALL_PAIRS) + 1 + len(helpers.FRED_SERIES) + 1
print(len(files), "files in data/raw, expected", expected)
assert len(files) == expected, "Some files are missing"